In [1]:
import pandas as pd
import os

## 1. Carrega os splits originais do CBF e o dataset completo

In [2]:
df_cbf_train = pd.read_csv('data_spliting/training.csv')
df_cbf_test  = pd.read_csv('data_spliting/testing.csv')
df_full      = pd.read_csv('ml-latest-small/ratings.csv')

print(f'CBF training : {df_cbf_train.shape[0]:>6} linhas (ratings >= 4.0)')
print(f'CBF testing  : {df_cbf_test.shape[0]:>6} linhas (ratings >= 4.0)')
print(f'Dataset full : {df_full.shape[0]:>6} linhas')

CBF training :  33351 linhas (ratings >= 4.0)
CBF testing  :  15229 linhas (ratings >= 4.0)
Dataset full : 100836 linhas


## 2. Extrai os ratings < 4.0 e aplica o mesmo split do divisao.ipynb

In [3]:
df_low = df_full[df_full['rating'] < 4.0].copy()
df_low = df_low.sort_values(by=['userId', 'timestamp'])

# Mesma lógica do divisao.ipynb: rank % 10 < 3 → teste, resto → treino
df_low['rank'] = df_low.groupby('userId').cumcount()
mask_test = df_low['rank'] % 10 < 3

df_low_test  = df_low[mask_test].drop(columns=['rank'])
df_low_train = df_low[~mask_test].drop(columns=['rank'])

print(f'Ratings < 4.0 → treino : {df_low_train.shape[0]:>6}')
print(f'Ratings < 4.0 → teste  : {df_low_test.shape[0]:>6}')

Ratings < 4.0 → treino :  35954
Ratings < 4.0 → teste  :  16302


## 3. Concatena e salva os novos splits híbridos

In [4]:
os.makedirs('data_spliting_hybrid', exist_ok=True)

df_hybrid_train = pd.concat([df_cbf_train, df_low_train], ignore_index=True)
df_hybrid_test  = pd.concat([df_cbf_test,  df_low_test],  ignore_index=True)

df_hybrid_train.to_csv('data_spliting_hybrid/training.csv', index=False)
df_hybrid_test.to_csv('data_spliting_hybrid/testing.csv',   index=False)

print(f'Híbrido training salvo: {df_hybrid_train.shape[0]:>6} linhas')
print(f'Híbrido testing  salvo: {df_hybrid_test.shape[0]:>6} linhas')
print(f'Total: {df_hybrid_train.shape[0] + df_hybrid_test.shape[0]} (esperado: {df_full.shape[0]})')

Híbrido training salvo:  69305 linhas
Híbrido testing  salvo:  31531 linhas
Total: 100836 (esperado: 100836)


## 4. Verificação: todos os registros originais do CBF estão no novo dataset?

In [5]:
# Chave de verificação: userId + movieId + rating + timestamp
def make_key(df):
    return set(zip(df['userId'], df['movieId'], df['rating'], df['timestamp']))

keys_cbf_train   = make_key(df_cbf_train)
keys_cbf_test    = make_key(df_cbf_test)
keys_hibrido_train = make_key(df_hybrid_train)
keys_hibrido_test  = make_key(df_hybrid_test)

faltando_train = keys_cbf_train - keys_hibrido_train
faltando_test  = keys_cbf_test  - keys_hibrido_test

print('=== Verificação de integridade ===')
print(f'Registros do CBF training ausentes no híbrido: {len(faltando_train)}')
print(f'Registros do CBF testing  ausentes no híbrido: {len(faltando_test)}')

if len(faltando_train) == 0 and len(faltando_test) == 0:
    print('\n✓ OK — todos os registros originais do CBF estão presentes nos novos splits.')
else:
    print('\n✗ ATENÇÃO — registros faltando! Verifique acima.')

=== Verificação de integridade ===
Registros do CBF training ausentes no híbrido: 0
Registros do CBF testing  ausentes no híbrido: 0

✓ OK — todos os registros originais do CBF estão presentes nos novos splits.


## 5. Resumo final

In [6]:
print('=== Resumo dos splits ===')
print(f'\n[Original CBF — apenas ratings >= 4.0]')
print(f'  training : {df_cbf_train.shape[0]:>6} linhas')
print(f'  testing  : {df_cbf_test.shape[0]:>6} linhas')

print(f'\n[Adicionados — ratings < 4.0, mesmo critério de split]')
print(f'  training : {df_low_train.shape[0]:>6} linhas')
print(f'  testing  : {df_low_test.shape[0]:>6} linhas')

print(f'\n[Híbrido final — data_spliting_hybrid/]')
print(f'  training : {df_hybrid_train.shape[0]:>6} linhas')
print(f'  testing  : {df_hybrid_test.shape[0]:>6} linhas')

print(f'\nDistribuição de ratings no training híbrido:')
print(df_hybrid_train['rating'].value_counts().sort_index())

print(f'\nDistribuição de ratings no testing híbrido:')
print(df_hybrid_test['rating'].value_counts().sort_index())

=== Resumo dos splits ===

[Original CBF — apenas ratings >= 4.0]
  training :  33351 linhas
  testing  :  15229 linhas

[Adicionados — ratings < 4.0, mesmo critério de split]
  training :  35954 linhas
  testing  :  16302 linhas

[Híbrido final — data_spliting_hybrid/]
  training :  69305 linhas
  testing  :  31531 linhas

Distribuição de ratings no training híbrido:
rating
0.5      915
1.0     1938
1.5     1272
2.0     5214
2.5     3862
3.0    13644
3.5     9109
4.0    18377
4.5     5852
5.0     9122
Name: count, dtype: int64

Distribuição de ratings no testing híbrido:
rating
0.5     455
1.0     873
1.5     519
2.0    2337
2.5    1688
3.0    6403
3.5    4027
4.0    8441
4.5    2699
5.0    4089
Name: count, dtype: int64
